# 2. Avec les LLMs

In [38]:
import json
import requests
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen3:8b"  # adapte si besoin

# ---- Schéma de sortie ----
class Experience(BaseModel):
    title: Optional[str] = None
    company: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    description: Optional[str] = None
    skills: List[str] = Field(default_factory=list)

class Education(BaseModel):
    degree: Optional[str] = None
    school: Optional[str] = None
    location: Optional[str] = None
    start_date: Optional[str] = None
    end_date: Optional[str] = None
    details: Optional[str] = None

class CVExtract(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    location: Optional[str] = None
    links: List[str] = Field(default_factory=list)
    summary: Optional[str] = None
    skills: List[str] = Field(default_factory=list)
    languages: List[str] = Field(default_factory=list)
    experiences: List[Experience] = Field(default_factory=list)
    education: List[Education] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)

def _extract_json_object(text: str) -> str:
    """
    Récupère le 1er objet JSON { ... } dans une réponse qui pourrait contenir du texte autour.
    """
    start = text.find("{")
    if start == -1:
        raise ValueError("Aucun '{' trouvé dans la réponse modèle.")

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i+1]
    raise ValueError("JSON incomplet: impossible de fermer les accolades.")

def call_ollama_extract(cv_text: str) -> CVExtract:
    system = (
        "Tu es un extracteur d'informations de CV. "
        "Retourne UNIQUEMENT un JSON strict, sans markdown, sans explications. "
        "Si un champ est inconnu, mets null ou une liste vide. "
        "N'invente rien."
    )

    user = f"""
Voici le texte d'un CV. Extrais les champs au format JSON suivant:

{{
  "name": string|null,
  "email": string|null,
  "phone": string|null,
  "location": string|null,
  "links": [string],
  "summary": string|null,
  "skills": [string],
  "languages": [string],
  "experiences": [
    {{
      "title": string|null,
      "company": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "description": string|null,
      "skills": [string]
    }}
  ],
  "education": [
    {{
      "degree": string|null,
      "school": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "details": string|null
    }}
  ],
  "certifications": [string]
}}

Texte CV:
\"\"\"{cv_text}\"\"\"
"""

    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "options": {"temperature": 0.0}
    }

    r = requests.post(OLLAMA_URL, json=payload, timeout=180)
    r.raise_for_status()
    data = r.json()
    content = data["message"]["content"]

    json_str = _extract_json_object(content)
    obj = json.loads(json_str)

    try:
        return CVExtract.model_validate(obj)
    except ValidationError as e:
        raise ValueError(f"JSON retourné invalide vs schéma: {e}") from e


In [39]:
from pypdf import PdfReader

def pdf_to_text(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    parts = []
    for i, page in enumerate(reader.pages):
        txt = page.extract_text() or ""
        # petite normalisation
        txt = txt.replace("\x00", "").strip()
        if txt:
            parts.append(f"\n\n--- PAGE {i+1} ---\n{txt}")
    return "\n".join(parts).strip()

cv_text = pdf_to_text("pdfs/cv_2.pdf")
print("Longueur texte extrait:", len(cv_text))
print(cv_text[:1200])


2026-01-15 14:42:02,272 - WARNING - Ignoring wrong pointing object 7 0 (offset 0)
2026-01-15 14:42:02,273 - WARNING - Ignoring wrong pointing object 26 0 (offset 0)
2026-01-15 14:42:02,273 - WARNING - Ignoring wrong pointing object 27 0 (offset 0)


Longueur texte extrait: 7518
--- PAGE 1 ---
Anouar Joual
Senior Data Scientist / AI engineer
(+33) 753178991   I     anojoual@gmail.com    I     Anouar joual    I     anouarjl
Data Scientist, passionné par l’innovation et l’apprentissage continu, maîtrisant les techniques de machine learning et d’analyse de 
données pour extraire des insights pertinents et concevoir des solutions à fort impact, guidées par les données.
Expérience Professionnelle                                                                                                                                                                                                           
Senior Data Scientist    	 	 	 	 	                                                                                                                               
Société Générale - Paris                                                                                                                                                                 

In [26]:
cv_text

'--- PAGE 1 ---\nAnouar Joual\nSenior Data Scientist / AI engineer\n(+33) 753178991   I     anojoual@gmail.com    I     Anouar joual    I     anouarjl\nData Scientist, passionné par l’innovation et l’apprentissage continu, maîtrisant les techniques de machine learning et d’analyse de \ndonnées pour extraire des insights pertinents et concevoir des solutions à fort impact, guidées par les données.\nExpérience Professionnelle                                                                                                                                                                                                           \nSenior Data Scientist    \t \t \t \t \t                                                                                                                               \nSociété Générale - Paris                                                                                                                                                                                

In [27]:
import re
from IPython.display import Markdown, display

def display_markdown(md_text: str):
    display(Markdown(md_text))

def normalize_ocr_line(line: str) -> str:
    """
    Recompose les mots quand chaque lettre est séparée par des espaces.
    Ex: 'D a t a  S c i e n t i s t' -> 'Data Scientist'
    """
    line = line.strip()

    # Si la ligne contient beaucoup de lettres séparées par des espaces
    if re.search(r"(?:[A-Za-z]\s){3,}", line):
        # On enlève les espaces entre lettres
        line = re.sub(r"(?<=\b[A-Za-z])\s(?=[A-Za-z]\b)", "", line)

    # Nettoyage espaces multiples
    line = re.sub(r"\s{2,}", " ", line)
    return line



norm_text = normalize_ocr_line(cv_text)
display_markdown(norm_text)



--- PAGE 1 ---
Anouar Joual
Senior Data Scientist / AI engineer
(+33) 753178991 I anojoual@gmail.com I Anouar joual I anouarjl
Data Scientist, passionné par l’innovation et l’apprentissage continu, maîtrisant les techniques de machine learning et d’analyse de données pour extraire des insights pertinents et concevoir des solutions à fort impact, guidées par les données.
Expérience Professionnelle Senior Data Scientist Société Générale - Paris Fév 2024 • Aujourd’hui - Détection automatisée de conversations non autorisées ou suspectes à l’aide de l’IA et du NLP.
- Préparation et prétraitement de grands volumes de données conversationnelles pour l’analyse sémantique.
- Entraînement des modèles LLM pour la classiﬁcation et la détection des patterns dans les conversations.
- Évaluation et optimisation rigoureuse des modèles via des boucles de ﬁne-tuning itératives.
- Mise en place de la transcription audio avec Whisper OpenAI pour convertir les messages vocaux en texte structuré.
- Développement de pipelines multilingues basés sur Whisper pour la détection automatique de la langue et la traduction du contenu transcrit. Stack technique: Python, Pytorch, Tensorﬂow, Spacy, Pandas, Jupyter-Lab, Transformers, HuggingFace,GIT, Linux, Jira.
Ingénieur Machine learning Orange - Paris Juin 2021 • Fév 2024 - Conception et développement d’un chatbot intelligent basé sur Rasa et BERT pour automatiser les interactions internes.
- Implémentation de la reconnaissance d’intention (NLP) et de la gestion de dialogue pour des réponses contextualisées.
- Intégration d’API pour la récupération de données en temps réel et l’automatisation des tâches.
- Réalisation de tests end-to-end et itérations successives aﬁn d’améliorer la précision et l’ergonomie du chatbot.
- Construction d’un pipeline CI/CD avec Git, Docker et tests automatisés pour le déploiement continu.
- Structuration et gestion des données conversationnelles avec MongoDB.
Stack technique: Python, Rasa, Pandas, Scikit-Learn, Spacy, MongoDB, CI/CD, GCP, APIs, GitLab, Linux, SQL, Jira.
Data scientist Operanka Associates - Paris Mars 2019 • Juin 2021 - Développement d’un chatbot IA dédié à l’automatisation de l’expertise réglementaire, intégrant des modèles NLP.
- Implémentation d’un modèle CNN de détection de mots-clés pour la reconnaissance vocale en temps réel, améliorant l’accessibilité du chatbot.
- Déploiement de microservices backend via Docker et AWS, garantissant performance et haute disponibilité.
- Conception d’un pipeline CI/CD avec GitHub Actions et AWS CodePipeline pour l’automatisation du build, des tests et du déploiement.
- Développement d’embeddings BERT pour l’analyse de similarité sémantique entre documents réglementaires.
- Expérimentation de différentes métriques de similarité et d’embeddings de phrases basés sur des Transformers pour optimiser la précision des comparaisons.
Stack technique: Python, Rasa, Pandas, Scikit-Learn, MongoDB, CI/CD, AWS, Transformers, Docker, Flask, Bitbucket, Linux, Jira.
Stagiaire Data scientist Nanovare - Lyon FR
Fév 2018 • Juillet 2018 - Développement de modèles de machine learning pour le diagnostic de la fertilité masculine.
- Prétraitement, nettoyage, corrélation et visualisation de données pour en extraire les insights clés. --- PAGE 2 ---
- Conception d’un modèle CNN de classiﬁcation d’images pour l’analyse de morphologie spermatique, améliorant la précision du diagnostic.
- Conception, entraînement et ﬁne-tuning de l’architecture CNN pour assurer robustesse et généralisation.
Stack technique: Python, Numpy, Pandas, Scikit-Learn, Visualisation, OpenCV, GIT, Linux.
Stagiaire Machine learning
LIMS Lab - Morocco
Fév 2017 • Juillet 2017 - Conception d’un système de classi ﬁcation du comportement du conducteur basé sur l’apprentissage supervisé et non supervisé.
- Entraînement de modèles prédictifs pour la détection et la classi ﬁcation des patterns de conduite, contribuant à l’analyse de la sécurité routière.
- Développement et déploiement d’une API Flask intégrant le modèle IA dans une application réelle.
- Optimisation des performances via feature engineering, réglage d’hyperparamètres et évaluation itérative. Stack technique: Python, Numpy, Pandas, Scikit-Learn, Flask,Jupyter, GIT, Linux.
Formation
Master M2 en Artiﬁcial Intelligence Université Claude Bernard Lyon1
2017 - 2018
Master en Big data analytics Maroc
Faculté des sciences - Fès
2015 - 2017 Licence en Mathématiques et Physique Faculté des sciences - Fès
2012 - 2015 Compétences Python, Scikit learn, Transformers, HuggingFace, Tensorﬂow, Visualization, Pandas, Numpy, MongoDB, Docker, Git, Spacy, NLTK, Pytorch, AWS, Github, MySQL, Flask, JupyterLab,EDA, Machine learning, Big data, LLM, NLP, Data processing, Agile, Jira, Linux, RASA, Time Management, Adaptability, Problem-Solving, Team spirit, Langues Arabe (langue maternelle) • Français (professionnel) • Anglais (professionnel) Projets Personnels - Assistant IA Génératif basé sur le RAG pour l’analyse documentaire : système utilisant un modèle Retrieval-Augmented Generation (RAG) pour interroger des documents PDF d’entreprise, identi ﬁer les sections pertinentes et générer des réponses contextuelles.
- Détection du port du masque facial avec YOLOv4 et Darknet.
- Application mobile multilingue “Scanner & Translate” utilisant OCR + NLP pour la classi ﬁcation et la traduction de texte dans les images.
- Système de recommandation de ﬁlms combinant ﬁltrage collaboratif et approche basée sur le contenu.
Centres d’Intérêt Apprentissage continu • Documentaires & podcasts • Cultures • Voyage • Randonnée • Fitness

In [11]:
# dérbuitage 

import re

def fix_spaced_numbers(text: str) -> str:
    """
    0 7 / 2 0 2 4 -> 07/2024
    2 0 2 2 -> 2022
    AZ - 9 0 0 -> AZ-900
    """
    # Fix dates MM/YYYY
    text = re.sub(r"(\d)\s(\d)\s*/\s*(\d)\s(\d)\s(\d)\s(\d)",
                  r"\1\2/\3\4\5\6", text)
    # Fix years YYYY
    text = re.sub(r"(\d)\s(\d)\s(\d)\s(\d)", r"\1\2\3\4", text)
    # Fix codes with spaced digits (AZ - 9 0 0 -> AZ-900 ; IA - 1 0 2 -> IA-102)
    text = re.sub(r"\b([A-Za-z]{2,})\s*-\s*(\d)\s(\d)\s(\d)\b", r"\1-\2\3\4", text)
    return text

def normalize_spaces(text: str) -> str:
    # Remplace espaces multiples
    text = re.sub(r"[ \t]{2,}", " ", text)
    # Corrige quelques collages fréquents
    text = text.replace("basedsearch", "based search")
    text = text.replace("forvisualization", "for visualization")
    return text.strip()

def clean_ocr_blob(raw: str) -> str:
    t = raw.replace("\\n", "\n")  # au cas où tu as des \n littéraux
    t = fix_spaced_numbers(t)
    t = normalize_spaces(t)
    return t


In [12]:
# Re-segmentation en lignes + détection sections
def to_lines(text: str) -> list[str]:
    """
    Transforme le bloc en lignes “logiques” via heuristiques :
    - démarre une nouvelle ligne avant: Tools:, EDUCATION, SKILLS, CERTIFICATIONS, WORK EXPERIENCE, HOBBIES
    - démarre une nouvelle ligne avant un pattern de poste courant + " - "
    - met les dates sur leur propre ligne
    """
    t = text

    # Ajoute sauts de ligne avant sections
    section_tokens = [
        "EDUCATION", "SKILLS & EXPERTISE", "CERTIFICATIONS", "WORK EXPERIENCE", "HOBBIES & INTERESTS"
    ]
    for tok in section_tokens:
        t = re.sub(rf"\s*{re.escape(tok)}\s*", f"\n{tok}\n", t)

    # Saut de ligne avant Tools :
    t = re.sub(r"\s*(Tools\s*:)", r"\n\1", t, flags=re.IGNORECASE)

    # Saut de ligne avant postes (heuristique)
    t = re.sub(r"\s*(Data Scientist\s*-\s*)", r"\n\1", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*(Data Engineer\s*-\s*)", r"\n\1", t, flags=re.IGNORECASE)

    # Dates sur ligne dédiée (MM/YYYY - ...)
    t = re.sub(r"\s*(\d{2}/\d{4}\s*-\s*(?:\d{2}/\d{4}|Present)(?:\s*\(.*?\))?)", r"\n\1\n", t)

    # Nettoyage des doubles points " . ."
    t = re.sub(r"\.\s*\.", ".", t)

    # Split + clean
    lines = []
    for line in t.splitlines():
        line = line.strip()
        if not line:
            continue
        lines.append(line)

    return lines


def classify_sections(lines: list[str]) -> dict[str, list[str]]:
    """
    Regroupe les lignes dans des sections génériques.
    """
    sections = {
        "HEADER": [],
        "WORK EXPERIENCE": [],
        "EDUCATION": [],
        "SKILLS & EXPERTISE": [],
        "CERTIFICATIONS": [],
        "HOBBIES & INTERESTS": [],
        "OTHER": [],
    }

    current = "HEADER"
    for line in lines:
        upper = line.upper()
        if upper in sections and upper != "HEADER":
            current = upper
            continue
        sections[current].append(line)

    return sections


In [13]:
# Génération markdown propre

def bullets_from_sentences(text: str) -> list[str]:
    """
    Découpe une ligne longue en points, et transforme en bullets.
    """
    # Split sur ". " en gardant sens
    parts = [p.strip() for p in re.split(r"\.\s+", text) if p.strip()]
    bullets = []
    for p in parts:
        p = p.rstrip(".")
        if len(p) >= 4:
            bullets.append(p)
    return bullets

def sections_to_markdown(sections: dict[str, list[str]]) -> str:
    md = []

    # Page header if present
    header_lines = sections.get("HEADER", [])
    # Enlève le "--- PAGE 1 ---" si présent
    header_lines = [l for l in header_lines if not re.match(r"---\s*PAGE\s*\d+\s*---", l, re.I)]
    if header_lines:
        # Essaye de détecter un nom (ligne "NOM de la personne" ou similaire)
        # sinon on met un titre générique
        md.append("# CV (extraction OCR nettoyée)\n")

    # WORK EXPERIENCE
    if sections.get("WORK EXPERIENCE"):
        md.append("## Work Experience\n")
        exp_lines = sections["WORK EXPERIENCE"]

        # Si OCR a mis des expériences avant le token "WORK EXPERIENCE", elles seront dans HEADER.
        # Donc on fusionne aussi les expériences qu'on détecte dans HEADER.
        candidates = sections.get("HEADER", []) + exp_lines

        i = 0
        while i < len(candidates):
            line = candidates[i]

            # Détecte un bloc expérience : "Data Scientist - ..."
            if re.search(r"\b(Data Scientist|Data Engineer)\b\s*-\s*", line, re.I):
                md.append(f"### {line}")

                # Cherche une date juste après
                if i + 1 < len(candidates) and re.search(r"\d{2}/\d{4}\s*-\s*", candidates[i+1]):
                    md.append(f"**{candidates[i+1]}**")
                    i += 1

                # Tools
                if i + 1 < len(candidates) and re.match(r"Tools\s*:", candidates[i+1], re.I):
                    md.append(f"**{candidates[i+1]}**")
                    i += 1

                # Description jusqu’au prochain job/section
                desc = []
                j = i + 1
                while j < len(candidates):
                    nxt = candidates[j]
                    if re.search(r"\b(Data Scientist|Data Engineer)\b\s*-\s*", nxt, re.I):
                        break
                    if nxt.upper() in ("EDUCATION","SKILLS & EXPERTISE","CERTIFICATIONS","HOBBIES & INTERESTS","WORK EXPERIENCE"):
                        break
                    # Ignore contacts bruités
                    if re.search(r"gmail|mail|num[eé]ro|linkedin", nxt, re.I):
                        j += 1
                        continue
                    desc.append(nxt)
                    j += 1

                # Ajoute bullets
                long_text = " ".join(desc).strip()
                if long_text:
                    for b in bullets_from_sentences(long_text):
                        md.append(f"- {b}")

                md.append("")  # blank line
                i = j
                continue

            i += 1

    # EDUCATION
    if sections.get("EDUCATION"):
        md.append("## Education\n")
        for l in sections["EDUCATION"]:
            # coupe grossièrement par " - 2022" etc.
            if re.search(r"\s-\s\d{4}\b", l):
                md.append(f"- {l}")
            else:
                md.append(f"- {l}")
        md.append("")

    # SKILLS
    if sections.get("SKILLS & EXPERTISE"):
        md.append("## Skills & Expertise\n")
        block = " ".join(sections["SKILLS & EXPERTISE"])
        # split par catégories "Programming languages :", etc.
        block = re.sub(r"\s*:\s*", ": ", block)
        items = re.split(r"\.\s*", block)
        for it in items:
            it = it.strip()
            if it:
                md.append(f"- {it}")
        md.append("")

    # CERTIFICATIONS
    if sections.get("CERTIFICATIONS"):
        md.append("## Certifications\n")
        block = " ".join(sections["CERTIFICATIONS"])
        items = re.split(r"\.\s*", block)
        for it in items:
            it = it.strip()
            if it:
                md.append(f"- {it}")
        md.append("")

    # HOBBIES
    if sections.get("HOBBIES & INTERESTS"):
        md.append("## Hobbies & Interests\n")
        # souvent collé sans séparateurs -> split simple par capitales/espaces
        block = " ".join(sections["HOBBIES & INTERESTS"]).strip()
        # Ex: TravelingPilatesRunningCategory B -> ajoute espaces avant majuscules
        block = re.sub(r"([a-z])([A-Z])", r"\1 \2", block)
        # split
        for it in re.split(r"\s{1,}", block):
            it = it.strip()
            if it and it.upper() not in ("HOBBIES", "&", "INTERESTS"):
                md.append(f"- {it}")
        md.append("")

    return "\n".join(md).strip()


In [14]:
from IPython.display import Markdown, display


cleaned = clean_ocr_blob(norm_text)
lines = to_lines(cleaned)
sections = classify_sections(lines)
md = sections_to_markdown(sections)

display(Markdown(md))


# CV (extraction OCR nettoyée)

## Education

- M 2 MoSEF ( Data science ) - 2022 Universit é Panth é on Sorbonne Bachelor ’ s Degree in ETI ( Statistics ) - 2020 Universit é Paris - Est Cr é teil PACES ( First year of medical studies ) - 2017 Universit é Paris - DiderotScientific Baccalaur é at - 2016 Lyc é e Rosa ParksFeriel OULHADJ DATA SCIENTIST

## Skills & Expertise

- Programming languages: Python , Pyspark , SQL
- Data science: NLP , Generative AI , Forecasting , Deep learning , Machinelearning , Statistiques
- MLOps: Docker , MLflow ( Model Registry ) , CI / CD ( GitHub Actions , Azure Pipelines ) , Hugging Face
- Cloud: Azure ( Azure Databricks , Azuredata factory , Azure OpenAI )
- APIs - Frontend: FastAPI , Gradio , Streamlit
- Version Control: GitHub , Azure DevOps
- Reporting: Power BILanguages: English ( C 1 ) , Spanish ( B 1 )

## Certifications

- Data science : text mining , machinelearning , deep learning
- Microsoft : AZ-900 , IA-102 SAS : Economic data analytics

## Hobbies & Interests

- Traveling
- Pilates
- Running
- Category
- B

In [17]:
import re

# -----------------------------
# Génération markdown propre (générique)
# -----------------------------

def bullets_from_sentences(text: str) -> list[str]:
    """
    Découpe un bloc de texte en phrases et les transforme en bullets.
    """
    parts = [p.strip() for p in re.split(r"\.\s+", text) if p.strip()]
    bullets = []
    for p in parts:
        p = p.rstrip(".")
        if len(p) >= 4:
            bullets.append(p)
    return bullets


# Heuristiques génériques
DATE_RANGE_RE = re.compile(
    r"\b("
    r"\d{2}/\d{4}\s*-\s*(?:\d{2}/\d{4}|PRESENT)"  # 07/2024 - Present
    r"|"
    r"\d{4}\s*-\s*(?:\d{4}|PRESENT)"             # 2021 - 2024
    r")\b",
    re.I
)

TOOLS_RE = re.compile(r"^(Tools|Outils|Stack)\s*:", re.I)

SECTION_CANON = {
    "WORK EXPERIENCE": "Work Experience",
    "PROFESSIONAL EXPERIENCE": "Work Experience",
    "EXPERIENCE": "Work Experience",
    "EDUCATION": "Education",
    "SKILLS & EXPERTISE": "Skills & Expertise",
    "SKILLS": "Skills & Expertise",
    "CERTIFICATIONS": "Certifications",
    "HOBBIES & INTERESTS": "Hobbies & Interests",
    "INTERESTS": "Hobbies & Interests",
}

def is_section_header(line: str) -> bool:
    up = line.strip().upper()
    if up in SECTION_CANON:
        return True
    # Heuristique : titres courts tout en majuscules
    if 3 <= len(up) <= 40 and re.fullmatch(r"[A-ZÀ-ÖØ-Ý0-9 &/.-]+", up):
        # évite de classer des phrases normales en headers
        if up.count(" ") <= 6:
            return True
    return False


def extract_experiences_generic(candidates: list[str]) -> list[dict]:
    """
    Détecte des blocs d'expériences de manière générique :
    - une ligne avec une plage de dates sert d'ancre
    - l'entête est la ligne juste avant (si elle n'est pas une section)
    - puis on agrège tools + description jusqu'au prochain bloc dates/section
    """
    exps = []
    date_idxs = [i for i, l in enumerate(candidates) if DATE_RANGE_RE.search(l)]

    for idx_pos, di in enumerate(date_idxs):
        header = candidates[di - 1].strip() if di - 1 >= 0 else "Experience"
        if is_section_header(header) or DATE_RANGE_RE.search(header):
            header = "Experience"

        exp = {"header": header, "dates": candidates[di].strip(), "tools": None, "desc": []}

        j = di + 1

        # Tools juste après la date (souvent le cas)
        if j < len(candidates) and TOOLS_RE.match(candidates[j].strip()):
            exp["tools"] = candidates[j].strip()
            j += 1

        # borne de fin = avant prochain bloc date
        end = (date_idxs[idx_pos + 1] - 1) if (idx_pos + 1 < len(date_idxs)) else (len(candidates) - 1)

        while j <= end:
            nxt = candidates[j].strip()

            if not nxt:
                j += 1
                continue

            if is_section_header(nxt):
                break

            # Ignore lignes contacts/bruit
            if re.search(r"gmail|e-?mail|mail\b|téléphone|telephone|linkedin|github", nxt, re.I):
                j += 1
                continue

            exp["desc"].append(nxt)
            j += 1

        exps.append(exp)

    return exps


def sections_to_markdown(sections: dict[str, list[str]]) -> str:
    md = []

    # Header
    header_lines = sections.get("HEADER", [])
    header_lines = [l for l in header_lines if not re.match(r"---\s*PAGE\s*\d+\s*---", l, re.I)]
    if header_lines:
        md.append("# CV (extraction OCR nettoyée)\n")

    # -----------------------------
    # WORK EXPERIENCE (GÉNÉRIQUE)
    # -----------------------------
    # On scanne TOUT ce qui pourrait contenir des expériences (sans dépendre du token WORK EXPERIENCE)
    candidates = []
    for k in ("HEADER", "OTHER", "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE", "EXPERIENCE"):
        candidates += sections.get(k, [])

    exps = extract_experiences_generic(candidates)

    if exps:
        md.append("## Work Experience\n")
        for e in exps:
            md.append(f"### {e['header']}")
            md.append(f"**{e['dates']}**")
            if e["tools"]:
                md.append(f"**{e['tools']}**")

            long_text = " ".join(e["desc"]).strip()
            if long_text:
                for b in bullets_from_sentences(long_text):
                    md.append(f"- {b}")
            md.append("")

    # -----------------------------
    # EDUCATION
    # -----------------------------
    if sections.get("EDUCATION"):
        md.append("## Education\n")
        for l in sections["EDUCATION"]:
            l = l.strip()
            if l:
                md.append(f"- {l}")
        md.append("")

    # -----------------------------
    # SKILLS
    # -----------------------------
    skills_key = "SKILLS & EXPERTISE" if sections.get("SKILLS & EXPERTISE") else ("SKILLS" if sections.get("SKILLS") else None)
    if skills_key:
        md.append("## Skills & Expertise\n")
        block = " ".join(sections[skills_key])
        block = re.sub(r"\s*:\s*", ": ", block)
        items = [it.strip() for it in re.split(r"\.\s*", block) if it.strip()]
        for it in items:
            md.append(f"- {it}")
        md.append("")

    # -----------------------------
    # CERTIFICATIONS
    # -----------------------------
    if sections.get("CERTIFICATIONS"):
        md.append("## Certifications\n")
        block = " ".join(sections["CERTIFICATIONS"])
        items = [it.strip() for it in re.split(r"\.\s*", block) if it.strip()]
        for it in items:
            md.append(f"- {it}")
        md.append("")

    # -----------------------------
    # HOBBIES
    # -----------------------------
    hobbies_key = "HOBBIES & INTERESTS" if sections.get("HOBBIES & INTERESTS") else ("INTERESTS" if sections.get("INTERESTS") else None)
    if hobbies_key:
        md.append("## Hobbies & Interests\n")
        block = " ".join(sections[hobbies_key]).strip()
        block = re.sub(r"([a-z])([A-Z])", r"\1 \2", block)  # TravelingPilates -> Traveling Pilates
        for it in re.split(r"\s{1,}", block):
            it = it.strip()
            if it and it.upper() not in ("HOBBIES", "&", "INTERESTS"):
                md.append(f"- {it}")
        md.append("")

    return "\n".join(md).strip()


In [19]:
import re

# -----------------------------
# 1) Helpers
# -----------------------------

def bullets_from_sentences(text: str) -> list[str]:
    """Découpe un bloc de texte en phrases et les transforme en bullets."""
    parts = [p.strip() for p in re.split(r"\.\s+", text) if p.strip()]
    return [p.rstrip(".") for p in parts if len(p.rstrip(".")) >= 4]


# Regex plus tolérante:
# - accepte "PresentTools" (donc pas de \b strict après Present)
# - accepte MM/YYYY - MM/YYYY|Present
# - accepte YYYY - YYYY|Present
DATE_RANGE_RE = re.compile(
    r"("
    r"\d{2}/\d{4}\s*-\s*(?:\d{2}/\d{4}|Present)"
    r"|"
    r"\d{4}\s*-\s*(?:\d{4}|Present)"
    r")",
    re.I
)

TOOLS_RE = re.compile(r"^(Tools|Outils|Stack)\s*:", re.I)

SECTION_CANON = {
    "WORK EXPERIENCE": "Work Experience",
    "PROFESSIONAL EXPERIENCE": "Work Experience",
    "EXPERIENCE": "Work Experience",
    "EDUCATION": "Education",
    "SKILLS & EXPERTISE": "Skills & Expertise",
    "SKILLS": "Skills & Expertise",
    "CERTIFICATIONS": "Certifications",
    "HOBBIES & INTERESTS": "Hobbies & Interests",
    "INTERESTS": "Hobbies & Interests",
}

def is_section_header(line: str) -> bool:
    up = line.strip().upper()
    if up in SECTION_CANON:
        return True
    # titres courts en majuscules
    if 3 <= len(up) <= 50 and re.fullmatch(r"[A-ZÀ-ÖØ-Ý0-9 &/.-]+", up):
        if up.count(" ") <= 8:
            return True
    return False


# -----------------------------
# 2) Pré-traitement CRUCIAL (générique)
# -----------------------------

def preprocess_lines_for_cv(lines: list[str]) -> list[str]:
    """
    Corrige les collages OCR fréquents de façon générique :
    - "PresentTools" -> "Present\nTools"
    - met les plages de dates sur une ligne dédiée (même si collées)
    - sépare aussi les headers collés (WORK EXPERIENCEHOBBIES -> WORK EXPERIENCE\nHOBBIES)
    """
    out = []
    for line in lines:
        s = line.strip()
        if not s:
            continue

        # 1) Sépare les headers collés (générique sur ces tokens connus)
        s = re.sub(r"(WORK EXPERIENCE)(HOBBIES\s*&\s*INTERESTS)", r"\1\n\2", s, flags=re.I)
        s = re.sub(r"(WORK EXPERIENCE)(HOBBIES)", r"\1\n\2", s, flags=re.I)
        s = re.sub(r"(LinkedIn)(EDUCATION)", r"\1\n\2", s, flags=re.I)

        # 2) "PresentTools" / "PresentOutils" / "PresentStack" -> new line
        s = re.sub(r"(Present)(Tools\s*:)", r"\1\n\2", s, flags=re.I)
        s = re.sub(r"(Present)(Outils\s*:)", r"\1\n\2", s, flags=re.I)
        s = re.sub(r"(Present)(Stack\s*:)", r"\1\n\2", s, flags=re.I)

        # 3) Si une plage de dates est EMBARQUÉE dans une ligne, on l’isole:
        #    "... SCC 07/2024 - PresentTools : ..." ->
        #    "... SCC\n07/2024 - Present\nTools : ..."
        def _split_dates(m: re.Match) -> str:
            return f"\n{m.group(1)}\n"

        s = DATE_RANGE_RE.sub(_split_dates, s)

        # 4) Force un saut avant Tools/Outils/Stack si collé au texte
        s = re.sub(r"\s*(Tools\s*:)", r"\n\1", s, flags=re.I)
        s = re.sub(r"\s*(Outils\s*:)", r"\n\1", s, flags=re.I)
        s = re.sub(r"\s*(Stack\s*:)", r"\n\1", s, flags=re.I)

        # 5) Split en sous-lignes et push
        for sub in s.split("\n"):
            sub = re.sub(r"\s{2,}", " ", sub).strip()
            if sub:
                out.append(sub)

    return out


# -----------------------------
# 3) Extraction d'expériences (générique via dates)
# -----------------------------

def extract_experiences_generic(candidates: list[str]) -> list[dict]:
    """
    - repère les lignes qui sont des dates (DATE_RANGE_RE)
    - prend la ligne précédente comme header si plausible
    - récupère Tools + description jusqu’à la prochaine date / section
    """
    exps = []

    # On veut des "lignes date" : ligne qui CONTIENT une date et ne contient pas 1000 autres trucs
    date_idxs = []
    for i, l in enumerate(candidates):
        if DATE_RANGE_RE.search(l):
            # si la ligne contient aussi trop de texte, c'est souvent une ligne non splittée,
            # mais notre preprocess est censé l'avoir corrigée. On garde quand même.
            date_idxs.append(i)

    for idx_pos, di in enumerate(date_idxs):
        header = candidates[di - 1].strip() if di - 1 >= 0 else "Experience"
        if is_section_header(header) or DATE_RANGE_RE.search(header):
            header = "Experience"

        exp = {"header": header, "dates": candidates[di].strip(), "tools": None, "desc": []}

        j = di + 1

        # Tools juste après
        if j < len(candidates) and TOOLS_RE.match(candidates[j].strip()):
            exp["tools"] = candidates[j].strip()
            j += 1

        end = (date_idxs[idx_pos + 1] - 1) if (idx_pos + 1 < len(date_idxs)) else (len(candidates) - 1)

        while j <= end:
            nxt = candidates[j].strip()
            if not nxt:
                j += 1
                continue
            if is_section_header(nxt):
                break
            if re.search(r"gmail|e-?mail|mail\b|téléphone|telephone|linkedin|github", nxt, re.I):
                j += 1
                continue
            exp["desc"].append(nxt)
            j += 1

        exps.append(exp)

    return exps


# -----------------------------
# 4) Markdown
# -----------------------------

def sections_to_markdown(sections: dict[str, list[str]]) -> str:
    md = []

    header_lines = sections.get("HEADER", [])
    header_lines = [l for l in header_lines if not re.match(r"---\s*PAGE\s*\d+\s*---", l, re.I)]
    md.append("# CV (extraction OCR nettoyée)\n")

    # candidates = tout ce qui peut contenir l'expérience
    candidates = []
    for k in ("HEADER", "OTHER", "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE", "EXPERIENCE"):
        candidates += sections.get(k, [])

    # IMPORTANT: preprocess pour corriger les collages
    candidates = preprocess_lines_for_cv(candidates)

    exps = extract_experiences_generic(candidates)

    if exps:
        md.append("## Work Experience\n")
        for e in exps:
            md.append(f"### {e['header']}")
            md.append(f"**{e['dates']}**")
            if e["tools"]:
                md.append(f"**{e['tools']}**")

            long_text = " ".join(e["desc"]).strip()
            if long_text:
                for b in bullets_from_sentences(long_text):
                    md.append(f"- {b}")
            md.append("")
    else:
        # debug-friendly : tu verras si aucun match date
        md.append("## Work Experience\n")
        md.append("_Aucune expérience détectée (vérifie les formats de dates dans le texte OCR)._")
        md.append("")

    # Education
    if sections.get("EDUCATION"):
        md.append("## Education\n")
        for l in preprocess_lines_for_cv(sections["EDUCATION"]):
            md.append(f"- {l}")
        md.append("")

    # Skills
    skills_key = "SKILLS & EXPERTISE" if sections.get("SKILLS & EXPERTISE") else ("SKILLS" if sections.get("SKILLS") else None)
    if skills_key:
        md.append("## Skills & Expertise\n")
        block = " ".join(sections[skills_key])
        block = re.sub(r"\s*:\s*", ": ", block)
        for it in [x.strip() for x in re.split(r"\.\s*", block) if x.strip()]:
            md.append(f"- {it}")
        md.append("")

    # Certifications
    if sections.get("CERTIFICATIONS"):
        md.append("## Certifications\n")
        block = " ".join(sections["CERTIFICATIONS"])
        for it in [x.strip() for x in re.split(r"\.\s*", block) if x.strip()]:
            md.append(f"- {it}")
        md.append("")

    # Hobbies
    hobbies_key = "HOBBIES & INTERESTS" if sections.get("HOBBIES & INTERESTS") else ("INTERESTS" if sections.get("INTERESTS") else None)
    if hobbies_key:
        md.append("## Hobbies & Interests\n")
        block = " ".join(sections[hobbies_key]).strip()
        block = re.sub(r"([a-z])([A-Z])", r"\1 \2", block)
        for it in re.split(r"\s{1,}", block):
            it = it.strip()
            if it and it.upper() not in ("HOBBIES", "&", "INTERESTS"):
                md.append(f"- {it}")
        md.append("")

    return "\n".join(md).strip()


In [30]:
def build_sections(lines: list[str]) -> dict[str, list[str]]:
    sections = {
        "HEADER": [],
        "OTHER": [],
        "WORK EXPERIENCE": [],
        "EDUCATION": [],
        "SKILLS & EXPERTISE": [],
        "CERTIFICATIONS": [],
        "HOBBIES & INTERESTS": [],
    }

    current = "HEADER"

    for line in lines:
        up = line.upper()

        if up in sections:
            current = up
            continue

        sections[current].append(line)

    return sections


lines = to_lines(norm_text)
sections = build_sections(lines)
md = sections_to_markdown(sections)
display(Markdown(md))

# CV (extraction OCR nettoyée)

## Work Experience

### Master M2 en Artiﬁcial Intelligence Université Claude Bernard Lyon1
**2017 - 2018**

### Experience
**2015 - 2017**
- Licence en Mathématiques et Physique Faculté des sciences - Fès

### Licence en Mathématiques et Physique Faculté des sciences - Fès
**2012 - 2015**

In [29]:
import re

# -----------------------------
# 0) Regex
# -----------------------------
DATE_MMYYYY = r"\d{2}\s*/\s*\d{4}"          # 07/2024 (avec espaces tolérés)
DATE_YYYY   = r"\d{4}"                     # 2024
PRESENT     = r"(?:Present|Aujourd'hui|Current|Now)"
DATE_RANGE_RE = re.compile(
    rf"({DATE_MMYYYY}\s*-\s*(?:{DATE_MMYYYY}|{PRESENT})|{DATE_YYYY}\s*-\s*(?:{DATE_YYYY}|{PRESENT}))",
    re.I
)

TOOLS_ANCHOR_RE = re.compile(r"\b(Tools|Outils|Stack)\s*:\s*", re.I)

SECTION_TOKENS = [
    "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE", "EXPERIENCE",
    "EDUCATION",
    "SKILLS & EXPERTISE", "SKILLS",
    "CERTIFICATIONS",
    "HOBBIES & INTERESTS", "INTERESTS",
]

SECTION_SPLIT_RE = re.compile(r"(" + "|".join(re.escape(s) for s in SECTION_TOKENS) + r")", re.I)

CONTACT_NOISE_RE = re.compile(r"(gmail|e-?mail|mail\b|téléphone|telephone|linkedin|github)", re.I)


# -----------------------------
# 1) Fix chiffres espacés (si tu as encore 0 7 / 2 0 2 4)
# -----------------------------
def fix_spaced_digits(text: str) -> str:
    # 0 7 / 2 0 2 4 -> 07/2024
    text = re.sub(r"(\d)\s(\d)\s*/\s*(\d)\s(\d)\s(\d)\s(\d)", r"\1\2/\3\4\5\6", text)
    # 2 0 2 4 -> 2024
    text = re.sub(r"(\d)\s(\d)\s(\d)\s(\d)", r"\1\2\3\4", text)
    return text


# -----------------------------
# 2) Segmentation agressive en lignes
# -----------------------------
def segment_text_aggressively(raw: str) -> list[str]:
    """
    Transforme un bloc OCR (même en 1 seule ligne) en lignes exploitables :
    - isole les sections (EDUCATION, SKILLS...)
    - isole les plages de dates sur leur ligne
    - isole Tools: sur sa ligne
    - coupe en pseudo-phrases
    """
    t = raw.replace("\\n", "\n")
    t = fix_spaced_digits(t)
    t = re.sub(r"[ \t]{2,}", " ", t).strip()

    # Sépare sections collées
    t = SECTION_SPLIT_RE.sub(r"\n\1\n", t)

    # Force un saut de ligne avant Tools/Outils/Stack partout
    t = TOOLS_ANCHOR_RE.sub(r"\n\1: ", t)

    # Isole les plages de dates même si collées (PresentTools -> Present\nTools)
    t = re.sub(r"(Present)(?=(Tools|Outils|Stack)\s*:)", r"\1\n", t, flags=re.I)

    # Isole dates sur une ligne dédiée (en conservant le reste)
    t = DATE_RANGE_RE.sub(r"\n\1\n", t)

    # Coupe en "phrases" grossières : point, point-virgule
    t = re.sub(r"([.;])\s+", r"\1\n", t)

    # Split final
    lines = [re.sub(r"\s{2,}", " ", l).strip() for l in t.splitlines()]
    lines = [l for l in lines if l]

    return lines


# -----------------------------
# 3) Détection générique d'expériences via ancres de dates
# -----------------------------
def bullets_from_sentences(text: str) -> list[str]:
    parts = [p.strip() for p in re.split(r"\.\s+|\;\s+", text) if p.strip()]
    return [p.rstrip(".") for p in parts if len(p.rstrip(".")) >= 4]


def is_section_header(line: str) -> bool:
    up = line.strip().upper()
    if up in [s.upper() for s in SECTION_TOKENS]:
        return True
    # Titres courts en majuscules
    if 3 <= len(up) <= 60 and re.fullmatch(r"[A-ZÀ-ÖØ-Ý0-9 &/.-]+", up):
        return True
    return False


def extract_experiences_from_lines(lines: list[str]) -> list[dict]:
    """
    Heuristique robuste :
    - une expérience = (header probable) + (date range) + (tools optionnels) + (desc jusqu’au prochain date/section)
    """
    # Index des lignes date
    date_idxs = [i for i, l in enumerate(lines) if DATE_RANGE_RE.search(l)]

    exps = []
    for k, di in enumerate(date_idxs):
        # header = cherche en arrière la première ligne "non section" et non vide, pas une date
        header = "Experience"
        back = di - 1
        while back >= 0:
            cand = lines[back].strip()
            if is_section_header(cand):
                back -= 1
                continue
            if DATE_RANGE_RE.search(cand):
                back -= 1
                continue
            # on évite de prendre Tools: comme header
            if TOOLS_ANCHOR_RE.search(cand):
                back -= 1
                continue
            header = cand
            break

        exp = {"header": header, "dates": lines[di].strip(), "tools": None, "desc": []}

        j = di + 1
        if j < len(lines) and TOOLS_ANCHOR_RE.search(lines[j]):
            exp["tools"] = lines[j].strip()
            j += 1

        end = date_idxs[k + 1] if (k + 1 < len(date_idxs)) else len(lines)
        while j < end:
            cur = lines[j].strip()
            if not cur or is_section_header(cur):
                break
            if CONTACT_NOISE_RE.search(cur):
                j += 1
                continue
            exp["desc"].append(cur)
            j += 1

        exps.append(exp)

    return exps


# -----------------------------
# 4) Markdown final
# -----------------------------
def to_markdown_from_raw_ocr(raw: str, debug: bool = False) -> str:
    lines = segment_text_aggressively(raw)

    if debug:
        print("---- DEBUG: 40 premières lignes après segmentation ----")
        for l in lines[:40]:
            print(l)
        print("\nNb lignes avec dates:", sum(1 for l in lines if DATE_RANGE_RE.search(l)))

    exps = extract_experiences_from_lines(lines)

    md = ["# CV (extraction OCR nettoyée)\n"]

    if exps:
        md.append("## Work Experience\n")
        for e in exps:
            md.append(f"### {e['header']}")
            md.append(f"**{e['dates']}**")
            if e["tools"]:
                md.append(f"**{e['tools']}**")
            long_text = " ".join(e["desc"]).strip()
            if long_text:
                for b in bullets_from_sentences(long_text):
                    md.append(f"- {b}")
            md.append("")
    else:
        md.append("## Work Experience\n")
        md.append("_Aucune expérience détectée : aucune plage de dates reconnue dans le texte._\n")

    return "\n".join(md).strip()



md = to_markdown_from_raw_ocr(norm_text, debug=True) 
display(Markdown(md))



---- DEBUG: 40 premières lignes après segmentation ----
--- PAGE 1 ---
Anouar Joual
Senior Data Scientist / AI engineer
(+33) 753178991 I anojoual@gmail.com I Anouar joual I anouarjl
Data Scientist, passionné par l’innovation et l’apprentissage continu, maîtrisant les techniques de machine learning et d’analyse de données pour extraire des insights pertinents et concevoir des solutions à fort impact, guidées par les données.
Expérience Professionnelle Senior Data Scientist Société Générale - Paris Fév 2024 • Aujourd’hui - Détection automatisée de conversations non autorisées ou suspectes à l’aide de l’IA et du NLP.
- Préparation et prétraitement de grands volumes de données conversationnelles pour l’analyse sémantique.
- Entraînement des modèles LLM pour la classiﬁcation et la détection des patterns dans les conversations.
- Évaluation et optimisation rigoureuse des modèles via des boucles de ﬁne-tuning itératives.
- Mise en place de la transcription audio avec Whisper OpenAI pour conv

# CV (extraction OCR nettoyée)

## Work Experience

### Master M2 en Artiﬁcial Intelligence Université Claude Bernard Lyon1
**2017 - 2018**

### Master M2 en Artiﬁcial Intelligence Université Claude Bernard Lyon1
**2015 - 2017**
- Licence en Mathématiques et Physique Faculté des sciences - Fès

### Licence en Mathématiques et Physique Faculté des sciences - Fès
**2012 - 2015**

In [22]:
import re
from dataclasses import dataclass
from typing import List, Dict, Any

DATE_MMYYYY = r"\d{2}\s*/\s*\d{4}"
DATE_YYYY   = r"\d{4}"
PRESENT     = r"(?:Present|Aujourd'hui|Current|Now)"
DATE_RANGE_RE = re.compile(
    rf"({DATE_MMYYYY}\s*-\s*(?:{DATE_MMYYYY}|{PRESENT})|{DATE_YYYY}\s*-\s*(?:{DATE_YYYY}|{PRESENT}))",
    re.I
)

TOOLS_ANCHOR_RE = re.compile(r"^(Tools|Outils|Stack)\s*:\s*", re.I)
CONTACT_NOISE_RE = re.compile(r"(gmail|e-?mail|mail\b|téléphone|telephone|phone|linkedin|github)", re.I)

SECTION_TOKENS = [
    "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE", "EXPERIENCE",
    "EDUCATION",
    "SKILLS & EXPERTISE", "SKILLS",
    "CERTIFICATIONS",
    "HOBBIES & INTERESTS", "INTERESTS",
]
SECTION_SPLIT_RE = re.compile(r"(" + "|".join(re.escape(s) for s in SECTION_TOKENS) + r")", re.I)

def fix_spaced_digits(text: str) -> str:
    text = re.sub(r"(\d)\s(\d)\s*/\s*(\d)\s(\d)\s(\d)\s(\d)", r"\1\2/\3\4\5\6", text)
    text = re.sub(r"(\d)\s(\d)\s(\d)\s(\d)", r"\1\2\3\4", text)
    return text

def segment_text_aggressively(raw: str) -> list[str]:
    t = raw.replace("\\n", "\n")
    t = fix_spaced_digits(t)
    t = re.sub(r"[ \t]{2,}", " ", t).strip()

    t = SECTION_SPLIT_RE.sub(r"\n\1\n", t)
    t = re.sub(r"(Present)(?=(Tools|Outils|Stack)\s*:)", r"\1\n", t, flags=re.I)
    t = re.sub(r"\b(Tools|Outils|Stack)\s*:\s*", r"\n\1 : ", t, flags=re.I)
    t = DATE_RANGE_RE.sub(r"\n\1\n", t)
    t = re.sub(r"([.;])\s+", r"\1\n", t)

    lines = [re.sub(r"\s{2,}", " ", l).strip() for l in t.splitlines()]
    return [l for l in lines if l]

def bullets_from_sentences(text: str) -> list[str]:
    parts = [p.strip() for p in re.split(r"\.\s+|\;\s+", text) if p.strip()]
    return [p.rstrip(".") for p in parts if len(p.rstrip(".")) >= 4]

def is_section_header(line: str) -> bool:
    up = line.strip().upper()
    return up in [s.upper() for s in SECTION_TOKENS]

def extract_experiences_from_lines(lines: list[str]) -> list[dict]:
    date_idxs = [i for i, l in enumerate(lines) if DATE_RANGE_RE.search(l)]
    exps = []

    for k, di in enumerate(date_idxs):
        # header = première ligne non section/non date au-dessus
        header = "Experience"
        back = di - 1
        while back >= 0:
            cand = lines[back].strip()
            if is_section_header(cand) or DATE_RANGE_RE.search(cand) or TOOLS_ANCHOR_RE.search(cand):
                back -= 1
                continue
            header = cand
            break

        exp = {"header": header, "dates": lines[di].strip(), "tools": None, "desc": []}

        j = di + 1
        if j < len(lines) and TOOLS_ANCHOR_RE.match(lines[j].strip()):
            exp["tools"] = lines[j].strip()
            j += 1

        end = date_idxs[k + 1] if (k + 1 < len(date_idxs)) else len(lines)
        while j < end:
            cur = lines[j].strip()
            if not cur or is_section_header(cur):
                break
            if CONTACT_NOISE_RE.search(cur):
                j += 1
                continue
            exp["desc"].append(cur)
            j += 1

        exps.append(exp)

    return exps

def split_sections(lines: list[str]) -> dict[str, list[str]]:
    sections = {tok.upper(): [] for tok in SECTION_TOKENS}
    sections["OTHER"] = []
    current = "OTHER"
    for l in lines:
        up = l.strip().upper()
        if up in sections and up != "OTHER":
            current = up
            continue
        sections[current].append(l)
    return sections

def build_markdown(exps: list[dict], sections: dict[str, list[str]]) -> str:
    md = ["# CV (extraction OCR nettoyée)\n"]

    if exps:
        md.append("## Work Experience\n")
        for e in exps:
            md.append(f"### {e['header']}")
            md.append(f"**{e['dates']}**")
            if e["tools"]:
                md.append(f"**{e['tools']}**")
            long_text = " ".join(e["desc"]).strip()
            if long_text:
                for b in bullets_from_sentences(long_text):
                    md.append(f"- {b}")
            md.append("")
    else:
        md.append("## Work Experience\n_Aucune expérience détectée._\n")

    # Ajoute d'autres sections si présentes (sans dépendre d'un format parfait)
    if sections.get("EDUCATION"):
        edu = [l for l in sections["EDUCATION"] if not CONTACT_NOISE_RE.search(l)]
        if edu:
            md.append("## Education\n")
            for l in edu:
                md.append(f"- {l}")
            md.append("")

    if sections.get("SKILLS & EXPERTISE") or sections.get("SKILLS"):
        key = "SKILLS & EXPERTISE" if sections.get("SKILLS & EXPERTISE") else "SKILLS"
        sk = [l for l in sections[key] if not CONTACT_NOISE_RE.search(l)]
        if sk:
            md.append("## Skills & Expertise\n")
            block = " ".join(sk)
            block = re.sub(r"\s*:\s*", ": ", block)
            for it in [x.strip() for x in re.split(r"\.\s*", block) if x.strip()]:
                md.append(f"- {it}")
            md.append("")

    if sections.get("CERTIFICATIONS"):
        cert = [l for l in sections["CERTIFICATIONS"] if not CONTACT_NOISE_RE.search(l)]
        if cert:
            md.append("## Certifications\n")
            block = " ".join(cert)
            for it in [x.strip() for x in re.split(r"\.\s*", block) if x.strip()]:
                md.append(f"- {it}")
            md.append("")

    if sections.get("HOBBIES & INTERESTS") or sections.get("INTERESTS"):
        key = "HOBBIES & INTERESTS" if sections.get("HOBBIES & INTERESTS") else "INTERESTS"
        hb = [l for l in sections[key] if not CONTACT_NOISE_RE.search(l)]
        if hb:
            md.append("## Hobbies & Interests\n")
            block = " ".join(hb).strip()
            block = re.sub(r"([a-z])([A-Z])", r"\1 \2", block)
            for it in [x.strip() for x in re.split(r"\s+", block) if x.strip()]:
                if it.upper() not in ("HOBBIES", "&", "INTERESTS"):
                    md.append(f"- {it}")
            md.append("")

    return "\n".join(md).strip()

def extract_all(raw: str, debug_lines: int = 40) -> Dict[str, Any]:
    lines = segment_text_aggressively(raw)
    exps = extract_experiences_from_lines(lines)
    sections = split_sections(lines)
    md = build_markdown(exps, sections)

    # Debug optionnel
    if debug_lines is not None:
        print(f"---- DEBUG: {debug_lines} premières lignes après segmentation ----")
        for l in lines[:debug_lines]:
            print(l)
        print("\nNb lignes totales:", len(lines))
        print("Nb lignes avec dates:", sum(1 for l in lines if DATE_RANGE_RE.search(l)))
        print("Nb expériences détectées:", len(exps))

    return {
        "lines": lines,             # <- toutes les lignes segmentées
        "experiences": exps,        # <- extraction structurée
        "sections": sections,       # <- ce que le parser a trouvé par sections
        "markdown": md,             # <- rendu final
    }


In [28]:
out = extract_all(norm_text, debug_lines=40)

# markdown final
from IPython.display import Markdown, display
display(Markdown(out["markdown"]))


---- DEBUG: 40 premières lignes après segmentation ----
--- PAGE 1 ---
Anouar Joual
Senior Data Scientist / AI engineer
(+33) 753178991 I anojoual@gmail.com I Anouar joual I anouarjl
Data Scientist, passionné par l’innovation et l’apprentissage continu, maîtrisant les techniques de machine learning et d’analyse de données pour extraire des insights pertinents et concevoir des solutions à fort impact, guidées par les données.
Expérience Professionnelle Senior Data Scientist Société Générale - Paris Fév 2024 • Aujourd’hui - Détection automatisée de conversations non autorisées ou suspectes à l’aide de l’IA et du NLP.
- Préparation et prétraitement de grands volumes de données conversationnelles pour l’analyse sémantique.
- Entraînement des modèles LLM pour la classiﬁcation et la détection des patterns dans les conversations.
- Évaluation et optimisation rigoureuse des modèles via des boucles de ﬁne-tuning itératives.
- Mise en place de la transcription audio avec Whisper OpenAI pour conv

# CV (extraction OCR nettoyée)

## Work Experience

### Master M2 en Artiﬁcial Intelligence Université Claude Bernard Lyon1
**2017 - 2018**
- Master en Big data analytics Maroc Faculté des sciences - Fès

### Faculté des sciences - Fès
**2015 - 2017**
- Licence en Mathématiques et Physique Faculté des sciences - Fès

### Licence en Mathématiques et Physique Faculté des sciences - Fès
**2012 - 2015**
- - Détection du port du masque facial avec YOLOv4 et Darknet
- - Application mobile multilingue “Scanner & Translate” utilisant OCR + NLP pour la classi ﬁcation et la traduction de texte dans les images
- - Système de recommandation de ﬁlms combinant ﬁltrage collaboratif et approche basée sur le contenu
- Centres d’Intérêt Apprentissage continu • Documentaires & podcasts • Cultures • Voyage • Randonnée • Fitness

# 3. Modification LLM


In [32]:
import re

# --------- Bilingue: ancres & patterns ---------
PRESENT_WORDS = r"(?:Present|Current|Now|Aujourd'hui|Actuel|En cours|À ce jour)"

SECTION_TOKENS = [
    # Work experience
    "WORK EXPERIENCE", "PROFESSIONAL EXPERIENCE", "EXPERIENCE", "EXPERIENCES",
    "EXPÉRIENCE", "EXPÉRIENCES", "EXPÉRIENCE PROFESSIONNELLE", "EXPÉRIENCES PROFESSIONNELLES",
    "PARCOURS PROFESSIONNEL",

    # Education
    "EDUCATION", "EDUCATIONAL BACKGROUND",
    "FORMATION", "FORMATIONS", "ÉTUDES", "ETUDES", "DIPLÔMES", "DIPLOMES",

    # Skills
    "SKILLS", "SKILLS & EXPERTISE", "TECHNICAL SKILLS", "CORE SKILLS",
    "COMPÉTENCES", "COMPETENCES", "COMPÉTENCES & EXPERTISE", "COMPETENCES & EXPERTISE",

    # Certifications
    "CERTIFICATIONS", "CERTIFICATION",

    # Hobbies
    "HOBBIES", "HOBBIES & INTERESTS", "INTERESTS",
    "CENTRES D’INTÉRÊT", "CENTRES D'INTERET", "CENTRES D INTERET", "LOISIRS",
]

SECTION_SPLIT_RE = re.compile(r"(" + "|".join(re.escape(s) for s in SECTION_TOKENS) + r")", re.I)

TOOLS_ANCHOR_RE = re.compile(r"\b(Tools|Outils|Stack|Technos|Technologies|Environnement|Tech stack)\s*:\s*", re.I)

# Dates: MM/YYYY - MM/YYYY|Present (tolère espaces), et YYYY - YYYY|Present
DATE_RANGE_RE = re.compile(
    rf"("
    rf"\d{{2}}\s*/\s*\d{{4}}\s*-\s*(?:\d{{2}}\s*/\s*\d{{4}}|{PRESENT_WORDS})"
    rf"|"
    rf"\d{{4}}\s*-\s*(?:\d{{4}}|{PRESENT_WORDS})"
    rf")",
    re.I
)

# Contact noise FR/EN
CONTACT_NOISE_RE = re.compile(
    r"(gmail|e-?mail|mail\b|téléphone|telephone|phone|mobile|linkedin|github|portfolio)",
    re.I
)

def fix_spaced_digits(text: str) -> str:
    # 0 7 / 2 0 2 4 -> 07/2024
    text = re.sub(r"(\d)\s(\d)\s*/\s*(\d)\s(\d)\s(\d)\s(\d)", r"\1\2/\3\4\5\6", text)
    # 2 0 2 4 -> 2024
    text = re.sub(r"(\d)\s(\d)\s(\d)\s(\d)", r"\1\2\3\4", text)
    return text

def clean_for_llm_bilingual(raw: str) -> str:
    """
    Rend le texte OCR/PDF “LLM-friendly” et robuste FR/EN.
    """
    t = raw.replace("\\n", "\n")
    t = fix_spaced_digits(t)
    t = re.sub(r"[ \t]{2,}", " ", t).strip()

    # Séparer sections (même si collées)
    t = SECTION_SPLIT_RE.sub(r"\n\1\n", t)

    # "PresentTools" / "Aujourd'huiOutils" etc.
    t = re.sub(rf"({PRESENT_WORDS})(?=({TOOLS_ANCHOR_RE.pattern}))", r"\1\n", t, flags=re.I)

    # Isoler tools/technos
    t = TOOLS_ANCHOR_RE.sub(r"\nTools: ", t)  # on normalise l'ancre à "Tools:" pour simplifier

    # Isoler plages de dates
    t = DATE_RANGE_RE.sub(r"\n\\1\n", t)

    # Coupe sur ponctuation pour aérer
    t = re.sub(r"([.;])\s+", r"\1\n", t)

    # Nettoyage final
    lines = [re.sub(r"\s{2,}", " ", l).strip() for l in t.splitlines()]
    lines = [l for l in lines if l]
    return "\n".join(lines)


In [50]:
from pypdf import PdfReader

def pdf_to_text_pypdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    parts = []
    for i, page in enumerate(reader.pages):
        txt = page.extract_text()
        if txt:
            parts.append(f"\n--- PAGE {i+1} ---\n{txt}")
    return "\n".join(parts)


In [47]:
import requests

OLLAMA_BASE = "http://localhost:11434"  # base
MODEL = "qwen2.5vl:3b"                      # vérifie dans /api/tags

def ollama_call(system: str, user: str, timeout: int = 180, temperature: float = 0.0) -> str:
    """
    Appelle Ollama de manière robuste:
    - essaie /api/chat (messages)
    - fallback /api/generate (prompt)
    """
    # 1) try chat
    chat_payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        "options": {"temperature": temperature},
    }
    try:
        r = requests.post(f"{OLLAMA_BASE}/api/chat", json=chat_payload, timeout=timeout)
        if r.status_code == 200:
            return r.json()["message"]["content"]
    except requests.RequestException:
        pass

    # 2) fallback generate
    gen_payload = {
        "model": MODEL,
        "prompt": system.strip() + "\n\n" + user.strip(),
        "stream": False,
        "options": {"temperature": temperature},
    }
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=gen_payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Ollama error: {r.status_code} - {r.text}")
    data = r.json()
    return data.get("response", "")


import json
from pydantic import ValidationError

def call_ollama_extract(cv_text: str) -> CVExtract:
    system = (
        "Tu es un extracteur d'informations de CV. "
        "Le CV peut être en français ou en anglais (ou mixte). "
        "Retourne UNIQUEMENT un JSON strict conforme au schéma demandé, sans markdown, sans explications. "
        "Si un champ est inconnu, mets null ou une liste vide. "
        "N'invente rien. "
        "Pour les dates, préfère ISO 'YYYY-MM' si possible, sinon garde le texte tel quel."
    )

    user = f"""
Le texte ci-dessous est un CV (FR/EN). Extrais les champs au format JSON suivant:

{{
  "name": string|null,
  "email": string|null,
  "phone": string|null,
  "location": string|null,
  "links": [string],
  "summary": string|null,
  "skills": [string],
  "languages": [string],
  "experiences": [
    {{
      "title": string|null,
      "company": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "description": string|null,
      "skills": [string]
    }}
  ],
  "education": [
    {{
      "degree": string|null,
      "school": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "details": string|null
    }}
  ],
  "certifications": [string]
}}

Important:
- Ne déduis pas un email/téléphone s'ils ne sont pas clairement présents.
- Les sections peuvent s'appeler Formation/Education, Expérience/Work Experience, Compétences/Skills, etc.
- En cours/Present/Actuel => expérience en cours.

Texte CV:
\"\"\"{cv_text}\"\"\"
""".strip()

    content = ollama_call(system=system, user=user, timeout=180, temperature=0.0)

    json_str = _extract_json_object(content)
    obj = json.loads(json_str)

    try:
        return CVExtract.model_validate(obj)
    except ValidationError as e:
        raise ValueError(f"JSON retourné invalide vs schéma: {e}") from e

def extract_cv_with_llm_bilingual(raw_text: str) -> CVExtract:
    cleaned = clean_for_llm_bilingual(raw_text)
    return call_ollama_extract(cleaned)

pdf_text = pdf_to_text_pypdf("pdfs/cv.pdf")
result = extract_cv_with_llm_bilingual(pdf_text)
print(result.model_dump_json(indent=2, ensure_ascii=False))



{
  "name": "Feriel OuLhadj",
  "email": "f.oulhadj@gmail.com",
  "phone": null,
  "location": "Paris, France",
  "links": [
    "Linkedin"
  ],
  "summary": "Data Scientist with expertise in Natural Language Processing, Generative AI, and Machine Learning. Skilled in Python, PySpark, and SQL. Experience in developing AI agents, designing and deploying IoT solutions, and improving customer experience in branches.",
  "skills": [
    "Programming languages: Python, PySpark, SQL",
    "Data Science: Natural Language Processing, Generative AI, Forecasting, Deep Learning, Machine Learning, Statistics",
    "ML Ops: Docker, ML Flow, CI/CD, Hugging Face",
    "Cloud: Azure (Azure DataBricks, Azure Data Factory, Azure Open AI)",
    "APIs: Frontend: FastAPI, Gradle, Streamlit",
    "Reporting: Power BI",
    "Languages: English (C1), Spanish (B1)"
  ],
  "languages": [
    "English",
    "Spanish"
  ],
  "experiences": [
    {
      "title": "Data Scientist - SCC (Specialist Computer Company)

In [49]:
display(result)

CVExtract(name='Feriel OuLhadj', email='f.oulhadj@gmail.com', phone=None, location='Paris, France', links=['Linkedin'], summary='Data Scientist with expertise in Natural Language Processing, Generative AI, and Machine Learning. Skilled in Python, PySpark, and SQL. Experience in developing AI agents, designing and deploying IoT solutions, and improving customer experience in branches.', skills=['Programming languages: Python, PySpark, SQL', 'Data Science: Natural Language Processing, Generative AI, Forecasting, Deep Learning, Machine Learning, Statistics', 'ML Ops: Docker, ML Flow, CI/CD, Hugging Face', 'Cloud: Azure (Azure DataBricks, Azure Data Factory, Azure Open AI)', 'APIs: Frontend: FastAPI, Gradle, Streamlit', 'Reporting: Power BI', 'Languages: English (C1), Spanish (B1)'], languages=['English', 'Spanish'], experiences=[Experience(title='Data Scientist - SCC (Specialist Computer Company)', company='Hugging Face', location='Paris, France', start_date='09/2022', end_date='07/2024',

In [61]:
import requests
import json
import re
from typing import Optional

OLLAMA_BASE = "http://localhost:11434"
MODEL = "qwen2.5vl:7b"  # idéalement un instruct texte si possible

def ollama_chat_json(system: str, user: str, timeout: int = 600, temperature: float = 0.0) -> str:
    """
    Essaie de forcer une sortie JSON (Ollama 'format': 'json' si supporté),
    sinon fallback sur extraction d'objet JSON depuis le texte.
    """
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        "stream": False,
        # Beaucoup d'install Ollama supportent ça :
        "format": "json",
        "options": {
            "temperature": temperature,
            # utiles pour réduire la "créativité"
            "top_p": 0.9,
            "repeat_penalty": 1.05,
        },
    }

    try:
        r = requests.post(f"{OLLAMA_BASE}/api/chat", json=payload, timeout=timeout)
        if r.status_code == 200:
            content = r.json()["message"]["content"]
            # si format=json, content est souvent déjà un JSON string
            return content
    except requests.RequestException:
        pass

    # fallback generate
    gen_payload = {
        "model": MODEL,
        "prompt": system.strip() + "\n\n" + user.strip(),
        "stream": False,
        "options": {"temperature": temperature, "top_p": 0.9, "repeat_penalty": 1.05},
    }
    r = requests.post(f"{OLLAMA_BASE}/api/generate", json=gen_payload, timeout=timeout)
    if r.status_code != 200:
        raise RuntimeError(f"Ollama error: {r.status_code} - {r.text}")
    return r.json().get("response", "")


def extract_first_json_object(text: str) -> str:
    """
    Extrait le premier objet JSON {...} en faisant du brace matching,
    robuste aux textes avant/après.
    """
    start = text.find("{")
    if start == -1:
        raise ValueError("Aucun '{' trouvé dans la réponse du modèle.")

    depth = 0
    in_str = False
    escape = False
    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    return text[start:i+1]

    raise ValueError("Objet JSON non terminé (brace matching incomplet).")



EMAIL_RE = re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE)
PHONE_RE = re.compile(r"(\+?\d[\d\s().-]{7,}\d)")
URL_RE = re.compile(r"\bhttps?://[^\s<>\"']+|www\.[^\s<>\"']+\b", re.IGNORECASE)

def preextract_facts(raw: str) -> dict:
    emails = list(dict.fromkeys(EMAIL_RE.findall(raw)))
    phones = [m.group(1).strip() for m in PHONE_RE.finditer(raw)]
    # filtre léger pour éviter des faux positifs trop courts
    phones = [p for p in phones if sum(c.isdigit() for c in p) >= 9]
    phones = list(dict.fromkeys(phones))

    urls = []
    for m in URL_RE.finditer(raw):
        u = m.group(0)
        if u.lower().startswith("www."):
            u = "https://" + u
        urls.append(u)
    urls = list(dict.fromkeys(urls))

    return {"emails": emails, "phones": phones, "urls": urls}


def build_system_prompt() -> str:
    return (
        "Tu es un extracteur d'informations de CV (FR/EN). "
        "Tu dois retourner UNIQUEMENT un JSON strict (pas de markdown, pas de texte). "
        "Règles strictes: "
        "- N'invente rien. Si inconnu -> null ou []. "
        "- name = nom complet de la personne (PAS un titre: 'Data Scientist', 'Engineer', etc.). "
        "- company = organisation employeuse (PAS une école). "
        "- school = établissement de formation (PAS une entreprise). "
        "- start_date/end_date: ISO 'YYYY-MM' si possible, sinon texte tel quel. "
        "- Si 'Present/Actuel/En cours' -> end_date = null. "
        "- N'utilise que les infos présentes dans le texte."
    )

def build_user_prompt(cv_text: str, facts: dict) -> str:
    schema = r'''
{
  "name": string|null,
  "email": string|null,
  "phone": string|null,
  "location": string|null,
  "links": [string],
  "summary": string|null,
  "skills": [string],
  "languages": [string],
  "experiences": [
    {
      "title": string|null,
      "company": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "description": string|null,
      "skills": [string]
    }
  ],
  "education": [
    {
      "degree": string|null,
      "school": string|null,
      "location": string|null,
      "start_date": string|null,
      "end_date": string|null,
      "details": string|null
    }
  ],
  "certifications": [string]
}
'''.strip()

    return f"""
Extrais les champs au format JSON suivant (doit matcher exactement le schéma) :

{schema}

Faits déjà détectés (tu dois les réutiliser si cohérents) :
- emails_detected: {facts.get("emails", [])}
- phones_detected: {facts.get("phones", [])}
- urls_detected: {facts.get("urls", [])}

Contraintes importantes:
- Si plusieurs emails/tels existent, choisis le plus "principal" (souvent en en-tête), sinon prends le premier.
- links = toutes les URLs utiles (LinkedIn, GitHub, portfolio, etc.).
- name: cherche en priorité dans les 15 premières lignes du CV. Si tu n'es pas sûr -> null.
- Ne mets JAMAIS un intitulé de poste dans name.

Texte CV:
\"\"\"{cv_text}\"\"\"
""".strip()



In [62]:
from pydantic import ValidationError

def post_guardrails(obj: dict) -> dict:
    # Empêche "name" = job title
    bad_name_tokens = [
        "data scientist", "engineer", "developer", "consultant", "manager",
        "ingénieur", "développeur", "consultant", "chef", "product owner"
    ]
    name = (obj.get("name") or "").strip()
    if name:
        low = name.lower()
        if any(tok in low for tok in bad_name_tokens) or len(name.split()) == 1:
            # 1 mot seul = souvent ambigu (mais à toi de voir)
            obj["name"] = None

    # Injecte email/phone si manquant mais trouvable dans facts (optionnel)
    return obj


def call_ollama_extract(cv_text: str) -> "CVExtract":

    facts = preextract_facts(cv_text)

    system = build_system_prompt()
    user = build_user_prompt(cv_text, facts)

    content = ollama_chat_json(system=system, user=user, timeout=600, temperature=0.0)

    # 1) parse JSON
    try:
        json_str = content.strip()
        # si le modèle renvoie du texte, on extrait l'objet
        if not json_str.startswith("{"):
            json_str = extract_first_json_object(content)
        obj = json.loads(json_str)
    except Exception as e:
        raise ValueError(f"Réponse non-JSON exploitable: {e}\n---\n{content[:800]}") from e

    # 2) guardrails
    obj = post_guardrails(obj)

    # 3) validate pydantic
    try:
        return CVExtract.model_validate(obj)
    except ValidationError as e:
        # Self-heal: on redemande au modèle de corriger UNIQUEMENT le JSON
        repair_system = (
            "Tu es un correcteur JSON. "
            "Retourne UNIQUEMENT un JSON strict corrigé (pas de texte)."
        )
        repair_user = f"""
Le JSON suivant doit respecter exactement ce schéma et passer la validation Pydantic.
Erreurs de validation:
{str(e)}

JSON à corriger:
{json.dumps(obj, ensure_ascii=False)}
""".strip()

        repaired = ollama_chat_json(repair_system, repair_user, timeout=600, temperature=0.0)
        try:
            repaired_str = repaired.strip()
            if not repaired_str.startswith("{"):
                repaired_str = extract_first_json_object(repaired)
            repaired_obj = json.loads(repaired_str)
            repaired_obj = post_guardrails(repaired_obj)
            return CVExtract.model_validate(repaired_obj)
        except Exception as e2:
            raise ValueError(f"JSON retourné invalide vs schéma (même après repair): {e2}") from e


In [63]:
def extract_cv_with_llm_bilingual(raw_text: str) -> "CVExtract":
    cleaned = clean_for_llm_bilingual(raw_text)  # ta fonction existante
    return call_ollama_extract(cleaned)

def run_on_pdf(pdf_path: str) -> "CVExtract":
    pdf_text = pdf_to_text_pypdf(pdf_path)       # ta fonction existante
    result = extract_cv_with_llm_bilingual(pdf_text)
    return result


In [64]:
res = run_on_pdf("pdfs/cv.pdf")


KeyboardInterrupt: 

In [60]:
display(res)

CVExtract(name='Feriel OuLhadj', email='f.oulhadj@gmail.com', phone='0 6 . 6 2 . 7 7 . 1 1 . 6 3', location='Paris', links=['https://www.linkedin.com/in/f-oulhadj/'], summary='Data Scientist with expertise in Natural Language Processing, Generative AI, and Machine Learning. Skilled in Python, PySpark, and SQL. Experienced in developing AI agents, designing and deploying IoT solutions, and implementing data pipelines.', skills=['Programming languages: Python, PySpark, SQL', 'Data Science: Natural Language Processing, Generative AI, Forecasting, Deep Learning, Machine Learning, Statistics', 'ML Ops: Docker, ML Flow, CI/CD, Hugging Face', 'Cloud: Azure (Azure DataBricks, Azure Data Factory, Azure Open AI)', 'APIs: Frontend, FastAPI, Gradle, Streamlit', 'Reporting: Power BI', 'Languages: English (C1), Spanish (B1)'], languages=['English (C1)', 'Spanish (B1)'], experiences=[Experience(title='Data Scientist - SCC (Specialist Computer Company)', company='Specialist Computer Company', location